<a href="https://colab.research.google.com/github/nmatsumoto-lgtm/study-KIKAGAKU/blob/main/%E4%B8%8D%E5%8B%95%E7%94%A3%E4%BE%A1%E6%A0%BC%E4%BA%88%E6%B8%AC%E3%82%A2%E3%83%97%E3%83%AA_Ver_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [176]:
!pip -q install lightgbm==4.5.0 optuna==3.6.1 joblib pandas numpy scikit-learn

In [177]:
!pip -q install geopandas pyogrio shapely pyproj rtree

In [178]:
from __future__ import annotations
import json
from pathlib import Path
from dataclasses import dataclass


import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_log_error
from joblib import dump
from optuna.samplers import TPESampler

In [179]:
# ===== 再現性セット（最上部の import の後に）=====
import os, random
SEED = 42  # ← 好きな固定値（既存の SEED=0 があれば 0 のままでもOK）
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [180]:
import os, re
import numpy as np
import pandas as pd

# 1) アップした元CSVのパス（ファイル名は自分のに合わせて変更）
SRC = "/content/平均売買㎡単価推移_東京都中央区.csv"

# 2) 出力先（学習コードが読む場所）
DST_DIR = "data/external"
DST = f"{DST_DIR}/avg_psm.csv"
os.makedirs(DST_DIR, exist_ok=True)

# 3) 読み込み
df = pd.read_csv(SRC)

# 4) 列の推定（period or year/month, そして avg_psm_yen）
cols_lower = {c.lower(): c for c in df.columns}

def pick_price_col(df):
    # まず既に英名があればそれを使う
    if "avg_psm_yen" in cols_lower:
        return cols_lower["avg_psm_yen"]
    # 日本語っぽい列名の自動検出（「平均」「㎡」「単価」「売買」などを含む列）
    for c in df.columns:
        s = str(c)
        if ("㎡" in s or "平米" in s or "m2" in s.lower()) and ("単価" in s or "価格" in s or "売買" in s):
            return c
    # 予備
    for c in df.columns:
        if "psm" in c.lower():
            return c
    raise ValueError("平均㎡単価の列が見つかりません。'avg_psm_yen' または日本語列名（例: '平均㎡売買単価'）を用意してください。")

def to_num(x):
    # "1,647,000 円" みたいな表記を数値化
    return (str(x)
            .replace(",", "")
            .replace("円", "")
            .replace("¥", "")
            .strip())

price_col = pick_price_col(df)
df[price_col] = pd.to_numeric(df[price_col].map(to_num), errors="coerce")

# period or year/month を作成
if "period" in cols_lower:
    period_col = cols_lower["period"]
    df["period"] = pd.to_numeric(df[period_col], errors="coerce").astype("Int64")
else:
    # 年月の列名を推測（year, month / 年, 月, 年月など）
    def pick_year_col():
        for k in ["year", "年"]:
            if k in cols_lower: return cols_lower[k]
        for c in df.columns:
            if re.search(r"年", str(c)): return c
        return None
    def pick_month_col():
        for k in ["month", "月"]:
            if k in cols_lower: return cols_lower[k]
        for c in df.columns:
            if re.search(r"月", str(c)): return c
        return None
    ycol = pick_year_col()
    mcol = pick_month_col()
    if ycol is not None and mcol is not None:
        yy = pd.to_numeric(df[ycol], errors="coerce").astype("Int64")
        mm = pd.to_numeric(df[mcol], errors="coerce").astype("Int64")
        df["period"] = (yy*100 + mm).astype("Int64")
    else:
        # "YYYYMM" 形式の列（例: 年月, yyyymm）を探す
        c_ym = None
        for c in df.columns:
            s = str(c).lower()
            if "yyyymm" in s or "年月" in s:
                c_ym = c; break
        if c_ym is None:
            raise ValueError("period か year/month（または 年/月/年月）の列が見つかりません。")
        df["period"] = pd.to_numeric(df[c_ym], errors="coerce").astype("Int64")

# 5) 最低限の2列だけに整形して保存
out = df[["period", price_col]].rename(columns={price_col: "avg_psm_yen"}).dropna()
out["period"] = out["period"].astype(int)
out["avg_psm_yen"] = out["avg_psm_yen"].astype(float)
out = out.sort_values("period")

out.to_csv(DST, index=False, encoding="utf-8-sig")
print("Saved ->", DST)
print(out.head())

Saved -> data/external/avg_psm.csv
   period  avg_psm_yen
0  202406    1290000.0
1  202408    1330000.0
2  202410    1380000.0
3  202412    1440000.0
4  202502    1520000.0


In [181]:
import io, zipfile, requests, pandas as pd, numpy as np, geopandas as gpd
from shapely.geometry import Point, Polygon
from pathlib import Path

DATA_EXT = Path("data/external"); DATA_EXT.mkdir(parents=True, exist_ok=True)

def _download_zip(url: str) -> zipfile.ZipFile:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return zipfile.ZipFile(io.BytesIO(r.content))

def _save_gml_or_shp_zip_to_geojson(zf: zipfile.ZipFile, out_path: Path):
    # ZIP内の .gml または .shp 一式を探す → GeoDataFrame で読み → WGS84化 → GeoJSON保存
    tmpdir = Path("/tmp/ksj"); tmpdir.mkdir(exist_ok=True, parents=True)
    zf.extractall(tmpdir)
    cand = None
    for ext in (".shp", ".gml"):
        files = list(tmpdir.rglob(f"*{ext}"))
        if files:
            cand = files[0]; break
    if cand is None:
        raise RuntimeError("ZIP内に GML/SHP が見つかりません。")

    gdf = gpd.read_file(cand)
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    gdf.to_file(out_path, driver="GeoJSON")

def _mesh250_csv_url(pref_code: str) -> str:
    # 旧サイトの CSV 版（都道府県ごと）
    # 例: 東京(13) → 250m_mesh_suikei_2024_csv_13.zip
    code = int(pref_code)
    fname = f"250m_mesh_suikei_2024_csv_{code:02d}.zip"
    # 実ファイルは old データ配下（datalist ページで列挙）
    return f"https://nlftp.mlit.go.jp/ksj/old/data/m250r6/m250r6-24/{fname}"

def _mesh250_zip_to_geojson_points(zf: zipfile.ZipFile, out_path: Path):
    import re, unicodedata
    import pandas as pd, numpy as np, geopandas as gpd
    from shapely.geometry import Point

    # 1) 文字列で全読み（桁落ち＆DtypeWarning回避）
    dfs = []
    for name in zf.namelist():
        if name.lower().endswith(".csv"):
            with zf.open(name) as fp:
                dfs.append(pd.read_csv(fp, encoding="cp932", dtype=str, low_memory=False))
    if not dfs:
        raise RuntimeError("ZIP内にCSVが見つかりません（mesh250r6）")
    df = pd.concat(dfs, ignore_index=True)

    # 2) 列名を正規化（全角→半角、小文字化、空白や記号を除去）
    def norm(s: str) -> str:
        s = unicodedata.normalize("NFKC", str(s)).lower()
        s = re.sub(r"[ \t\r\n\(\)（）［］\[\]{}【】＜＞<>:：;；,、。.-]", "", s)
        return s
    col_norm = {c: norm(c) for c in df.columns}
    # 逆引き（正規化名 → 実列名）
    inv = {}
    for orig, n in col_norm.items():
        inv.setdefault(n, orig)

    # 3) 候補語彙で meshcode / lat / lon を探す
    mesh_keys = [
        "meshcode","mesh","基準地域メッシュコード","標準地域メッシュコード",
        "地域メッシュコード","地域メッシュ","メッシュコード","ﾒｯｼｭｺｰﾄﾞ","基準ﾒｯｼｭｺｰﾄﾞ"
    ]
    lat_keys  = ["lat","latitude","ido","緯度","中心緯度","代表点緯度","緯度代表点","中心点緯度"]
    lon_keys  = ["lon","lng","longitude","keido","経度","中心経度","代表点経度","経度代表点","中心点経度"]

    def find_by_keys(keys):
        for k in keys:
            nk = norm(k)
            if nk in inv:
                return inv[nk]
        # 正規化名が部分一致する列も拾う（例：中心点緯度度）
        for c,nc in col_norm.items():
            if any(norm(k) in nc for k in keys):
                return c
        return None

    mcol = find_by_keys(mesh_keys)
    latcol = find_by_keys(lat_keys)
    loncol = find_by_keys(lon_keys)

    # 4) 値パターンでも meshcode を補足（6〜10桁の数字が多い列を優先）
    if mcol is None:
        candidates = []
        for c in df.columns:
            s = df[c].astype(str).str.strip()
            ok = s.str.match(r"^\d{6,10}$", na=False)
            if ok.mean() > 0.6 and ok.sum() > 100:  # 閾値は経験則
                # ユニーク数が多いほどmeshcodeらしい
                candidates.append((c, ok.sum(), s[ok].nunique()))
        if candidates:
            candidates.sort(key=lambda t: (-t[1], -t[2]))
            mcol = candidates[0][0]

    # 5) 緯度経度が見つからなければ数値域で推定（緯度20–50, 経度120–155）
    def _pick_range(colnames, lo, hi):
        best = None; score = -1
        for c in df.columns:
            v = pd.to_numeric(df[c], errors="coerce")
            ratio = ((v >= lo) & (v <= hi)).mean()
            if ratio > score and ratio > 0.6:
                best, score = c, ratio
        return best
    if latcol is None:
        latcol = _pick_range(df.columns, 20, 50)
    if loncol is None:
        loncol = _pick_range(df.columns, 120, 155)

    # 6) 人口列の検出（“それっぽい”名前＋数値平均でスコア）
    def is_numeric_series(s):
        pd.to_numeric(s, errors="coerce"); return True
    num_cols = [c for c in df.columns if is_numeric_series(df[c])]

    def score_pop(c):
        name = col_norm[c]
        # 0合計は最下位に落とす
        vals = pd.to_numeric(df[c], errors="coerce")
        total = float(vals.fillna(0).sum())
        if total <= 0:
            return (9999, 0.0)

        # 名前ヒントの優先度（小さいほど有力）
        name_order = ["総人口","将来","pop","population","pt00","ptn"]
        name_score = min([i for i,k in enumerate(name_order) if k in name] + [100])

        # 合計が大きいほど有力（tie-breaker）
        return (name_score, -total)

    # def score_pop(c):
    #     name = col_norm[c]
    #     name_score = 100
    #     for i,h in enumerate(["pt00","ptn","総人口","将来","pop","population"]):
    #         if h in name:
    #             name_score = min(name_score, i)
    #     mean_val = pd.to_numeric(df[c], errors="coerce").mean(skipna=True)
    #     return (name_score, -float(mean_val if pd.notna(mean_val) else 0))
    if not num_cols:
        raise RuntimeError("人口値らしき数値列が見つかりません（mesh250r6）")
    popcol = sorted(num_cols, key=score_pop)[0]

    # 7) 緯度経度が使えるならそのまま点にする
    if latcol and loncol:
        lat = pd.to_numeric(df[latcol], errors="coerce")
        lon = pd.to_numeric(df[loncol], errors="coerce")
        mask = lat.between(20, 50) & lon.between(120, 155)
        sub = df.loc[mask, [popcol] + ([mcol] if mcol else [])].copy()
        sub.rename(columns={popcol:"pop", mcol:"meshcode"}, inplace=True)
        sub["pop"] = pd.to_numeric(sub["pop"], errors="coerce").fillna(0.0)
        gdf = gpd.GeoDataFrame(sub, geometry=gpd.points_from_xy(lon[mask], lat[mask]), crs=4326)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        gdf.to_file(out_path, driver="GeoJSON")
        return

    # 8) 最後の手段：meshcode から中心点を厳密に復元（6/7/8桁対応）
    if not mcol:
        raise RuntimeError("緯度経度も meshcode も見つかりません。CSVの列名を確認してください。")

    mesh = df[mcol].astype(str).str.strip()
    mesh = mesh.where(mesh.str.match(r"^\d{6,10}$", na=False)).dropna()
    if mesh.empty:
        raise RuntimeError("meshcode の値が空/不正でした。CSVの 'メッシュコード' 列をご確認ください。")

    def mesh_center_ll(m: str):
        # 1次(2/3°×1°) → 二次(1/12°×1/8°) → 500m(象限) → 250m(象限)
        a = int(m[0:2]); b = int(m[2:4]); y = int(m[4]); x = int(m[5])
        lat0 = a * (2/3); lon0 = 100 + b * 1.0
        dlat2 = (2/3) / 8.0; dlon2 = 1.0 / 8.0
        lat2  = lat0 + y * dlat2; lon2 = lon0 + x * dlon2
        dlat3, dlon3 = dlat2/2.0, dlon2/2.0
        dlat4, dlon4 = dlat3/2.0, dlon3/2.0
        lat, lon = lat2 + dlat2/2, lon2 + dlon2/2   # 1km中心
        if len(m) >= 7:
            q3 = int(m[6])
            lat = lat2 + (0 if q3 in (1,2) else dlat3) + dlat3/2
            lon = lon2 + (0 if q3 in (1,3) else dlon3) + dlon3/2
        if len(m) >= 8:
            q4 = int(m[7])
            lat = (lat - dlat3/2) + (0 if q4 in (1,2) else dlat4) + dlat4/2
            lon = (lon - dlon3/2) + (0 if q4 in (1,3) else dlon4) + dlon4/2
        return (lat, lon)

    centers = mesh.map(mesh_center_ll)
    lat = centers.map(lambda t: t[0]); lon = centers.map(lambda t: t[1])
    sub = df.loc[mesh.index, [mcol, popcol]].copy().rename(columns={mcol:"meshcode", popcol:"pop"})
    sub["pop"] = pd.to_numeric(sub["pop"], errors="coerce").fillna(0.0)
    gdf = gpd.GeoDataFrame(sub, geometry=gpd.points_from_xy(lon, lat), crs=4326)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(out_path, driver="GeoJSON")


In [182]:
SEED = 0
np.random.seed(SEED)

In [183]:
IN = Path("data/interim/train_tabular.csv")
OUT_DIR = Path("models"); OUT_DIR.mkdir(parents=True, exist_ok=True)

In [184]:
# === 追加（先頭の定数定義あたり）========================
IDX_CSV = Path("data/external/avg_psm.csv")  # 平均㎡単価CSV
USE_DEFLATE = True  # ← True: ②デフレート学習を有効化 / False: ①の特徴量化のみ
# =======================================================

In [185]:
# ===== 先に置くヘルパー（build_station_env_features より前のセルに）=====
import shutil, unicodedata
from pathlib import Path
import pandas as pd

DATA_EXT = Path("data/external"); DATA_EXT.mkdir(parents=True, exist_ok=True)

# 1) アップロードした stations_geo を確実に所定パスへコピー
src_candidates = [
    Path("/mnt/data/stations_geo .csv"),  # ←ファイル名にスペースがある版
    Path("/content/stations_geo .csv"),
    Path("/content/stations_geo.csv"),
    Path("stations_geo.csv"),
]
STATIONS_GEO_CSV = DATA_EXT / "stations_geo.csv"
for src in src_candidates:
    if src.exists():
        shutil.copy(src, STATIONS_GEO_CSV)
        print("Copied:", src, "→", STATIONS_GEO_CSV)
        break
else:
    raise FileNotFoundError("stations_geo.csv が見つかりません。Colabにアップ後、このセルを再実行してください。")

def load_stations_csv(path: Path) -> pd.DataFrame:
    import unicodedata, pandas as pd, numpy as np

    def norm_col(c):
        c = unicodedata.normalize("NFKC", str(c)).strip().lower()
        c = c.replace("：", ":")  # 念のため
        return c

    s = pd.read_csv(path)
    s.columns = [norm_col(c) for c in s.columns]

    # 正規化後の別名セット（全部 NFKC＋lower 前提で書く）
    aliases = {
        "station": ["station", "station_name", "最寄駅:名称", "最寄駅：名称", "駅名"],
        "lat":     ["lat", "lat.", "latitude", "緯度"],
        "lon":     ["lon", "lon.", "long", "longitude", "経度"],
    }

    # 見つかったらその列名を揃える
    rename_map = {}
    for std, cands in aliases.items():
        for c in s.columns:
            if c in [norm_col(x) for x in cands]:
                rename_map[c] = std
                break
    s = s.rename(columns=rename_map)

    if not {"station","lat","lon"} <= set(s.columns):
        raise ValueError(f"stations_geo.csv に station/lat/lon がありません。現在の列: {list(s.columns)}")

    s = s[["station","lat","lon"]].copy()
    s["station"] = s["station"].astype(str).map(lambda x: unicodedata.normalize("NFKC", x).strip())
    s["lat"] = pd.to_numeric(s["lat"], errors="coerce")
    s["lon"] = pd.to_numeric(s["lon"], errors="coerce")
    s = s.dropna(subset=["station","lat","lon"]).reset_index(drop=True)

    # ヘッダ行混入の除去
    bad = s["station"].str.lower().isin(["station","最寄駅:名称","最寄駅：名称","駅名"])
    s = s.loc[~bad].reset_index(drop=True)
    return s

Copied: /content/stations_geo .csv → data/external/stations_geo.csv


In [186]:
# ====== 地理ユーティリティ（追記）=============================================
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from shapely import area as shp_area
import shutil
import unicodedata

DATA_EXT = Path("data/external"); DATA_EXT.mkdir(parents=True, exist_ok=True)
DATA_RAW = Path("data/raw"); DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_INT = Path("data/interim"); DATA_INT.mkdir(parents=True, exist_ok=True)

# アップロードした候補（スペース入りのファイル名にも対応）
src_candidates = [
    Path("/mnt/data/stations_geo .csv"),
    Path("/content/stations_geo .csv"),
    Path("/content/stations_geo.csv"),
]
dst = DATA_EXT / "stations_geo.csv"

for src in src_candidates:
    if src.exists():
        shutil.copy(src, dst)
        print("Copied:", src, "→", dst)
        break
else:
    raise FileNotFoundError("stations_geo.csv が見つかりません。Colab にアップ後に再実行してください。")

STATIONS_GEO_CSV = dst  # ← 以後はこのパスを使う
print("Using stations file:", STATIONS_GEO_CSV)


FLOOD_GJ = Path("data/external/flood_zone.geojson")
POP_GJ   = Path("data/external/popmesh_250m.geojson")
ENV_BY_ST = DATA_INT / "env_features_by_station.csv"

def fetch_ksj_layer(layer: str, pref_code: str, out_path: Path) -> Path:
    """KSJレイヤーを公式サイトからDL→GeoJSON化して保存（A49は最新版優先でフォールバック）"""
    if out_path.exists():
        return out_path

    if layer.upper() == "A49":
        # まずは最新版A49-20を試す。ダメなら古い版を順にフォールバック。
        pref = int(pref_code)
        versions = ["20", "19", "18", "17", "16", "15", "14", "13", "12", "11"]
        last_err = None
        for ver in versions:
            url = f"https://nlftp.mlit.go.jp/ksj/gml/data/A49/A49-{ver}/A49-{ver}_{pref:02d}_GML.zip"
            try:
                zf = _download_zip(url)  # ← 既存の関数を再利用
                _save_gml_or_shp_zip_to_geojson(zf, out_path)
                return out_path
            except Exception as e:
                last_err = e
                continue
        raise RuntimeError(f"A49 ダウンロード失敗（試行: {versions}）。最後のエラー: {last_err}")

    if layer.lower() == "mesh250r6":
        # 2024年度CSV（都道府県別）
        url = _mesh250_csv_url(pref_code)  # 既存の関数をそのまま使用
        zf = _download_zip(url)
        _mesh250_zip_to_geojson_points(zf, out_path)
        return out_path

    raise ValueError(f"unknown layer: {layer}")

def _find_vector(path_hint: Path) -> Path | None:
    """
    path_hint が:
      - 既存のファイル（.geojson / .shp など）ならそのまま返す
      - 既存のディレクトリなら中の .shp を優先探索、次点で .geojson
      - それ以外は <hint>.geojson / <hint>.shp を順に探す
    """
    # 1) そのものズバリ（拡張子付きの完全パス）
    if path_hint.exists() and path_hint.is_file():
        return path_hint

    # 2) ディレクトリなら中を探す（.shp優先）
    if path_hint.exists() and path_hint.is_dir():
        shp_list = list(path_hint.glob("*.shp"))
        if shp_list:
            return shp_list[0]
        gj_list = list(path_hint.glob("*.geojson"))
        if gj_list:
            return gj_list[0]
        return None

    # 3) 基準名（拡張子なし）として扱い、順に試す
    for cand in [path_hint.with_suffix(".geojson"), path_hint.with_suffix(".shp")]:
        if cand.exists():
            return cand
    return None


def _to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        # 日本の公的データは JGD2011/EPSG:6668 か平面直角 EPSG:66xx のことが多い
        # ここでは安全に WGS84 を明示して使えるよう、事前に crs を確認して設定してから reproject してください。
        # 不明なら WGS84 と仮定するのは危険なので、ここではそのまま通す。
        pass
    return gdf.to_crs(4326) if gdf.crs and gdf.crs.to_epsg() != 4326 else gdf

def prepare_flood_geojson(
    raw_hint: Path = DATA_RAW / "flood_zone",  # '.../flood_zone.shp' or '.../flood_zone.geojson' or ディレクトリ
    out_path: Path = FLOOD_GJ
) -> Path:
    """高潮浸水想定区域を WGS84 にして GeoJSON 出力。既に out_path があればスキップ。"""
    if out_path.exists():
        return out_path
    src = _find_vector(raw_hint)
    if src is None:
        raise FileNotFoundError(
            f"高潮データが見つかりません。次のいずれかを置いてください: {raw_hint}.geojson / {raw_hint}.shp / ディレクトリ（.shp一式）"
        )
    gdf = gpd.read_file(src)
    gdf = _to_wgs84(gdf)

    # 列の標準化（任意）：強度クラスなどがあれば名前を合わせておく
    # 候補: 'depth','想定浸水深','max_depth' など。ここでは存在する最初の列を使って 'flood_cls' に寄せる。
    depth_cols = [c for c in gdf.columns if str(c).lower() in ("depth","max_depth","浸水深","想定浸水深")]
    if depth_cols:
        gdf = gdf.rename(columns={depth_cols[0]: "flood_cls"})
    elif "flood_cls" not in gdf.columns:
        gdf["flood_cls"] = None

    gdf.to_file(out_path, driver="GeoJSON")
    return out_path

def prepare_popmesh_geojson(
    raw_hint: Path = DATA_RAW / "popmesh_250m",  # '.../popmesh_250m.shp' or '.geojson' or ディレクトリ
    out_path: Path = POP_GJ
) -> Path:
    """将来推計人口メッシュ（250m）を WGS84 にして GeoJSON 出力。既に out_path があればスキップ。"""
    if out_path.exists():
        return out_path
    src = _find_vector(raw_hint)
    if src is None:
        raise FileNotFoundError(
            f"人口メッシュが見つかりません。次のいずれかを置いてください: {raw_hint}.geojson / {raw_hint}.shp / ディレクトリ（.shp一式）"
        )
    gdf = gpd.read_file(src)
    gdf = _to_wgs84(gdf)

    # 人口列の標準化：最も“それっぽい”数値列を pop に採用（pop, PPL, 将来人口 など）
    num_cols = [c for c in gdf.columns if np.issubdtype(gdf[c].dtype, np.number)]
    pick = None
    for key in ["pop","population","将来人口","future_pop","PPL","PPL_2035","PPL_2040","PPL_2050"]:
        if key in gdf.columns:
            pick = key; break
    if pick is None:
        # 最後の保険：最大値が大きい数値列を人口と見なす
        if not num_cols:
            raise ValueError("人口っぽい数値列が見つかりません。'pop' 等の列を含めてください。")
        pick = sorted(num_cols, key=lambda c: float(gdf[c].fillna(0).max()), reverse=True)[0]
    gdf = gdf.rename(columns={pick: "pop"})
    gdf.to_file(out_path, driver="GeoJSON")
    return out_path

def build_station_env_features(
    stations_geo_csv: Path = STATIONS_GEO_CSV,
    flood_geojson: Path = FLOOD_GJ,
    pop_geojson: Path = POP_GJ,
    out_csv: Path = ENV_BY_ST,
    buffer_m: float = 400.0
) -> Path:
    """
    駅（緯度経度）→ 400mバッファ（EPSG:3857）
      ・高潮：重なり率 / 最近傍距離
      ・将来人口：ポイントならバッファ内合計、ポリゴンなら面積按分合計
    を計算し CSV を出力。
    """
    assert Path(stations_geo_csv).exists(), f"Not found: {stations_geo_csv}"
    assert Path(flood_geojson).exists(), f"Not found: {flood_geojson}"
    assert Path(pop_geojson).exists(),   f"Not found: {pop_geojson}"

    # ---- 駅点（WGS84）
    s = load_stations_csv(stations_geo_csv)
    if not {"station","lat","lon"} <= set(s.columns):
        raise ValueError("stations_geo.csv には 'station','lat','lon' 列が必要です。")
    g_st = gpd.GeoDataFrame(
        s,
        geometry=[Point(float(lon), float(lat)) for lat, lon in zip(s["lat"], s["lon"])],
        crs=4326
    )

    # ---- データ読み込み
    g_flood = gpd.read_file(flood_geojson)
    g_pop   = gpd.read_file(pop_geojson)   # ← 引数名なのでOK
    g_pop   = _ensure_pop_column(g_pop)

    # CRS を WGS84 に正規化
    for g in (g_st, g_flood, g_pop):
        if g.crs is None:
            # 公式配布は大抵 GML/GeoJSON で CRS が付く想定。None は危険なので明示要求。
            raise ValueError("入力データの CRS が不明です。WGS84(EPSG:4326) か、CRSを明示してください。")
        if g.crs.to_epsg() != 4326:
            g.to_crs(4326, inplace=True)

    # ---- 3857 へ変換（距離・面積はメートル系で計算）
    g_st_3857    = g_st.to_crs(3857)
    g_flood_3857 = g_flood.to_crs(3857)
    g_pop_3857   = g_pop.to_crs(3857)

    # ---- 駅 400m バッファ（ついでに全バッファを union してクリップ用に作成）
    g_st_3857["buf"] = g_st_3857.geometry.buffer(buffer_m)
    union_buf = g_st_3857["buf"].unary_union

    # 大幅な高速化：対象領域で事前クリップ
    if len(g_flood_3857):
        g_flood_3857 = gpd.clip(g_flood_3857, union_buf)
    # if len(g_pop_3857):
    #     g_pop_3857 = gpd.clip(g_pop_3857, union_buf)

    # ---- 高潮：重なり率（バッファに占める浸水面積割合） & 最近傍距離
    flood_ratio = []
    flood_idx   = []
    if len(g_flood_3857) > 0:
        flood_union = g_flood_3857.unary_union  # 距離計算用（高速）
        # 面積重なり（各バッファと flood の交差面積）
        # 空間インデックスで候補を絞る
        sidx_flood = g_flood_3857.sindex
        for geom_buf in g_st_3857["buf"].values:
            cand_ids = list(sidx_flood.query(geom_buf, predicate="intersects"))
            if not cand_ids:
                flood_ratio.append(0.0); flood_idx.append(0); continue
            cand = g_flood_3857.iloc[cand_ids]
            inter = cand.geometry.intersection(geom_buf)
            inter_area = float(np.sum([g.area for g in inter if not g.is_empty]))
            flood_idx.append(1 if inter_area > 0 else 0)
            flood_ratio.append(inter_area / float(geom_buf.area + 1e-9))
        # 最近傍距離（点→ポリゴン）
        dist_m = g_st_3857.geometry.distance(flood_union).fillna(1e6)
    else:
        flood_ratio = [0.0] * len(g_st_3857)
        flood_idx   = [0]   * len(g_st_3857)
        dist_m      = pd.Series([1e6] * len(g_st_3857), index=g_st_3857.index, dtype=float)

    # ---- 人口：ポイント or ポリゴンで分岐
    # pop 列を厳密に数値化
    if "pop" not in g_pop_3857.columns:
        raise ValueError("人口データに 'pop' 列がありません。fetch側で 'pop' に正規化してください。")
    g_pop_3857["pop"] = pd.to_numeric(g_pop_3857["pop"], errors="coerce").fillna(0.0).astype(float)

    pop_sum = []
    geom_types = set(g_pop_3857.geom_type.unique())
    sidx_pop = g_pop_3857.sindex if len(g_pop_3857) else None


    if geom_types <= {"Point"}:
        # 事前クリップ（高速&安定）
        g_pop_3857 = gpd.clip(g_pop_3857, union_buf)

        # バッファをGeoDataFrame化
        g_buf = gpd.GeoDataFrame(
            {"station": g_st_3857["station"].values},
            geometry=g_st_3857["buf"].values,
            crs=3857
        )

    joined = gpd.sjoin(
        g_pop_3857[["pop", "geometry"]],
        g_buf[["station", "geometry"]],
        how="inner",
        predicate="within",  # 点がバッファ内
    )

    # どの列名で駅名が入っているかを安全に特定
    st_col = next(c for c in ["station", "station_right", "station_left"] if c in joined.columns)

    pop_by_st = joined.groupby(st_col)["pop"].sum()

    # もとの駅順に並べて配列化
    pop_sum = [float(pop_by_st.get(st, 0.0)) for st in g_st_3857["station"].values]


    # ---- 出力
    env = pd.DataFrame({
        "station": g_st["station"].values,
        "flood_any": np.array(flood_idx, dtype=int),
        "flood_ratio": np.array(flood_ratio, dtype=float),
        "dist_flood_m": dist_m.values.astype(float),
        "pop400": np.array(pop_sum, dtype=float),
    })
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    env.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return out_csv


Copied: /content/stations_geo .csv → data/external/stations_geo.csv
Using stations file: data/external/stations_geo.csv


Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [187]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


,Registration State,Violation Description,count
0,CA,No Parking,50341
1,NJ,No Parking,50091
2,NY,Expired Meter,50409
3,TX,No Parking,50146


In [188]:
# === 追加：指数CSVを読み込み、月次(YYYYMM)に整えて返す =========
def _load_avg_psm_monthly(csv_path: Path) -> pd.DataFrame:
    idx = pd.read_csv(csv_path)

    # period が無ければ year,month から作成
    cols = {c.lower(): c for c in idx.columns}
    if "period" not in cols:
        ycol = cols.get("year"); mcol = cols.get("month")
        if ycol is None or mcol is None:
            raise ValueError("avg_psm.csv には period か year,month が必要です。")
        idx["period"] = idx[ycol].astype(int) * 100 + idx[mcol].astype(int)

    # 月次にリサンプリング（2か月置き→月次へ・線形補間）
    idx["ym"] = pd.to_datetime(idx["period"].astype(str), format="%Y%m")
    idx = (idx[["ym", "avg_psm_yen"]]
             .set_index("ym")
             .resample("MS")
             .interpolate("time")
             .ffill()
             .reset_index())
    idx["period"] = (idx["ym"].dt.year * 100 + idx["ym"].dt.month).astype(int)

    # MoM/YoYなど派生（任意）
    idx = idx.sort_values("period").reset_index(drop=True)
    idx["log_idx_psm"] = np.log(idx["avg_psm_yen"].astype(float))
    idx["idx_mom"] = idx["avg_psm_yen"].pct_change().fillna(0.0)
    idx["idx_yoy"] = idx["avg_psm_yen"].pct_change(12).fillna(0.0)
    return idx[["period", "avg_psm_yen", "log_idx_psm", "idx_mom", "idx_yoy"]]
# =========================================================

def compute_calibration(period_arr, idx_csv: Path, smooth_months: int = 2) -> float:
    """
    学習末月の外部平均㎡単価と最新月の外部平均㎡単価の比（移動平均で平滑化）を返す。
    deflate有無に関わらず、最終の円スケールに掛けるだけでOK。
    """
    try:
        idx = _load_avg_psm_monthly(idx_csv).sort_values("period")
        last_train = int(np.max(period_arr))

        p_train = idx.loc[idx["period"] <= last_train, "avg_psm_yen"]
        if len(p_train) == 0:
            return 1.0
        base = float(p_train.tail(smooth_months).mean())

        latest = float(idx["avg_psm_yen"].tail(smooth_months).mean())
        return latest / base if base > 0 else 1.0
    except Exception:
        return 1.0

In [189]:
# === 中央区 5エリア：駅→エリアの固定マップ ===
STATION_TO_AREA = {
    # ① 銀座・京橋
    "銀座": "銀座・京橋",
    "東銀座": "銀座・京橋",
    "銀座一丁目": "銀座・京橋",
    "京橋(東京)": "銀座・京橋",
    "宝町(東京)": "銀座・京橋",
    "新橋": "銀座・京橋",
    "汐留": "銀座・京橋",
    "東京": "銀座・京橋",  # 日本橋側に寄せたい場合は「日本橋・人形町」へ変更可

    # ② 日本橋・人形町
    "日本橋(東京)": "日本橋・人形町",
    "三越前": "日本橋・人形町",
    "新日本橋": "日本橋・人形町",
    "茅場町": "日本橋・人形町",
    "人形町": "日本橋・人形町",
    "水天宮前": "日本橋・人形町",
    "浜町": "日本橋・人形町",
    "東日本橋": "日本橋・人形町",
    "小伝馬町": "日本橋・人形町",
    "馬喰横山": "日本橋・人形町",
    "馬喰町": "日本橋・人形町",

    # ③ 月島・勝どき・晴海
    "月島": "月島・勝どき・晴海",
    "勝どき": "月島・勝どき・晴海",

    # ④ 築地・八丁堀
    "築地": "築地・八丁堀",
    "築地市場": "築地・八丁堀",
    "新富町(東京)": "築地・八丁堀",
    "八丁堀(東京)": "築地・八丁堀",
}

AREA_LIST = ["銀座・京橋", "日本橋・人形町", "月島・勝どき・晴海", "築地・八丁堀", "佃・新川・湊", "その他"]

def station_to_area(station_name: str) -> str:
    """駅名を5エリアに変換。未登録は 'その他'。"""
    if not isinstance(station_name, str) or station_name.strip() == "":
        return "その他"
    return STATION_TO_AREA.get(station_name.strip(), "その他")

In [190]:
# ============================
# 1) データ読み込み & 追加特徴量
# ============================

def _oof_target_median(df: pd.DataFrame, key: str, y: np.ndarray, n_splits: int = 5) -> np.ndarray:
    """キー列ごとの OOF 中央値エンコーディング（リーク防止）。
    戻り値は各行に対応する OOF 推定値（学習折ごとに学習外の中央値を使う）。
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(df), dtype=float)
    for tr_idx, va_idx in kf.split(df):
        med = (
            pd.DataFrame({key: df.iloc[tr_idx][key].values, "y": y[tr_idx]})
            .groupby(key)["y"].median()
        )
        oof[va_idx] = df.iloc[va_idx][key].map(med).fillna(med.median()).values
    return oof

def _oof_target_mean(df, key, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(df), dtype=float)
    for tr_idx, va_idx in kf.split(df):
        m = (pd.DataFrame({key: df.iloc[tr_idx][key].values, "y": y[tr_idx]})
             .groupby(key)["y"].mean())
        oof[va_idx] = df.iloc[va_idx][key].map(m).fillna(m.mean()).values
    return oof

def build_table_and_features() -> tuple[pd.DataFrame, list[str], list[str], list[str], list[int]]:
    df = pd.read_csv(IN)

    # ===== 対数ターゲット =====
    y = df["price_yen"].astype(float).values
    # ★ 追加：指数CSVを読み込み & マージ
    idx = _load_avg_psm_monthly(IDX_CSV)

    # 学習データ側 period が無ければ作成（あなたのCSVには既にありますが保険）
    if "period" not in df.columns and {"year","month"} <= set(df.columns):
        df["period"] = df["year"].astype(int)*100 + df["month"].astype(int)

    # マージ（相場水準をくっつける）
    df = df.merge(idx, on="period", how="left")
    # 欠損は前方/後方で埋める（学習期間内なら埋まります）
    for c in ["avg_psm_yen","log_idx_psm","idx_mom","idx_yoy"]:
        df[c] = df[c].fillna(method="ffill").fillna(method="bfill")

    # ★ ここが“②デフレート学習”の切替ポイント
    if USE_DEFLATE:
        # 相場水準(avg_psm)を引いた「相対価格（対数）」を学習
        y_log = np.log1p(y) - np.log(df["avg_psm_yen"].astype(float).values)
    else:
        # 従来どおり
        y_log = np.log1p(y)

    # # ===== 対数ターゲット =====
    # y = df["price_yen"].astype(float).values
    # y_log = np.log1p(y)

    # ===== 5エリアを駅から自動付与（入力は駅のまま）=====
    df["area_group"] = df["station"].astype(str).apply(station_to_area)

    # ===== 追加特徴量 =====
    df["access_score"] = 1.0 / (1.0 + df["walk_min"].astype(float))
    df["sqrt_area"]    = np.sqrt(np.clip(df["area_sqm"].astype(float), 0, None))
    df["log_area"]     = np.log1p(np.clip(df["area_sqm"].astype(float), 0, None))
    df["age_sqrt"]     = np.sqrt(np.clip(df["築年数"].astype(float), 0, None))
    df["area_x_access"]= df["area_sqm"].astype(float) * df["access_score"].astype(float)

    # ===== 頻度特徴量 =====
    station_cnt = df["station"].value_counts()
    layout_cnt  = df["layout"].value_counts()
    area_cnt    = df["area_group"].value_counts()

    df["station_count"] = df["station"].map(station_cnt).fillna(0).astype(int)
    df["layout_count"]  = df["layout"].map(layout_cnt).fillna(0).astype(int)
    df["area_count"]    = df["area_group"].map(area_cnt).fillna(0).astype(int)

     # ===== ここから追記：駅400m圏の環境特徴をJOIN =====
    env = pd.read_csv(ENV_BY_ST)  # 'station','flood_any','flood_ratio','dist_flood_m','pop400'
    df = df.merge(env, on="station", how="left")
    # 欠損があれば安全側で埋める
    df["flood_any"]     = df["flood_any"].fillna(0).astype(int)
    df["flood_ratio"]   = df["flood_ratio"].fillna(0.0)
    df["dist_flood_m"]  = df["dist_flood_m"].fillna(1e6)  # 非該当は遠く扱い
    df["pop400"]        = df["pop400"].fillna(0.0)

    # ===== OOF 目標エンコード（対数ターゲットで中央値）=====
    df["station_oof_median_log"] = _oof_target_median(df, "station",     y_log)
    df["layout_oof_median_log"]  = _oof_target_median(df, "layout",      y_log)
    df["area_oof_median_log"]    = _oof_target_median(df, "area_group",  y_log)

    # ===== OOF 目標エンコード（対数ターゲットで平均）=====  ←★ ここを追加
    df["station_oof_mean_log"] = _oof_target_mean(df, "station",    y_log)
    df["layout_oof_mean_log"]  = _oof_target_mean(df, "layout",     y_log)
    df["area_oof_mean_log"]    = _oof_target_mean(df, "area_group", y_log)


    # ===== 推論用に学習時統計を準備 =====
    station_med_map = (
        pd.DataFrame({"station": df["station"], "y_log": y_log})
        .groupby("station")["y_log"].median().to_dict()
    )
    layout_med_map = (
        pd.DataFrame({"layout": df["layout"], "y_log": y_log})
        .groupby("layout")["y_log"].median().to_dict()
    )
    area_med_map = (
        pd.DataFrame({"area_group": df["area_group"], "y_log": y_log})
        .groupby("area_group")["y_log"].median().to_dict()
    )

    # ←★ ここから “平均” を追加
    station_mean_map = (
        pd.DataFrame({"station": df["station"], "y_log": y_log})
        .groupby("station")["y_log"].mean().to_dict()
    )
    layout_mean_map = (
        pd.DataFrame({"layout": df["layout"], "y_log": y_log})
        .groupby("layout")["y_log"].mean().to_dict()
    )
    area_mean_map = (
        pd.DataFrame({"area_group": df["area_group"], "y_log": y_log})
        .groupby("area_group")["y_log"].mean().to_dict()
    )

    global_station_med = float(np.median(list(station_med_map.values()))) if len(station_med_map) else float(np.median(y_log))
    global_layout_med  = float(np.median(list(layout_med_map.values())))  if len(layout_med_map)  else float(np.median(y_log))
    global_area_med    = float(np.median(list(area_med_map.values())))    if len(area_med_map)    else float(np.median(y_log))

    # 参考：平均の“全体値”も欲しければ
    global_station_mean = float(np.mean(list(station_mean_map.values()))) if len(station_mean_map) else float(np.mean(y_log))
    global_layout_mean  = float(np.mean(list(layout_mean_map.values())))  if len(layout_mean_map)  else float(np.mean(y_log))
    global_area_mean    = float(np.mean(list(area_mean_map.values())))    if len(area_mean_map)    else float(np.mean(y_log))

    # ===== カテゴリ列（area を先頭に追加）=====
    feat_cat = ["area_group", "station", "layout"]

    # ===== 数値列 =====
    feat_num_base = ["walk_min", "築年数", "area_sqm"]
    feat_num_extra = [
        "access_score", "sqrt_area", "log_area", "age_sqrt",
        "area_x_access",
        "station_count", "layout_count", "area_count",
        "station_oof_median_log", "layout_oof_median_log", "area_oof_median_log",
        "station_oof_mean_log", "layout_oof_mean_log", "area_oof_mean_log",
        # ★ 追加：指数の特徴量（①）
        "log_idx_psm",   # 水準（対数）
        "idx_mom",       # 前月比
        "idx_yoy",       # 前年同月比
        # ---- ここから追加（環境特徴）----
        "flood_ratio",     # バッファ内の浸水割合（↑=リスク↑）
        "dist_flood_m",    # 浸水想定域までの距離（↑=安全）
        "pop400",          # 400m内の将来人口（↑=賑わい/需要）
    ]
    feat_num_all = feat_num_base + feat_num_extra

    # 単調制約を「名前で」安全に指定
    mono_dict = {name: 0 for name in feat_num_all}
    mono_dict.update({
        "walk_min": -1,      # 遠いほど↓
        "築年数": -1,         # 古いほど↓
        "area_sqm": +1,      # 広いほど↑
        # ← 派生も一貫させる
        "access_score": +1,  # 駅近スコアが高いほど↑（walk_minと反対符号に注意だが整合的）
        "sqrt_area": +1,
        "log_area": +1,
        "age_sqrt": -1,
         # 環境の直感的制約
        "flood_ratio": -1,     # 浸水割合が大きいほど価格↓
        "dist_flood_m": +1,    # 浸水域から遠いほど価格↑
        "pop400": +1,          # 人口密度が高いほど価格↑（一般に正だが市場によっては調整）

     })
    monotone_constraints = [0] * len(feat_cat) + [mono_dict[c] for c in feat_num_all]

    # # ===== 単調性制約（[cat..., num...] の順）=====
    # n_cat = len(feat_cat)
    # # walk_min↓, 築年数↓, area_sqm↑、その他は制約なし
    # mono_num = [-1, -1, +1] + [0] * (len(feat_num_all) - 3)
    # monotone_constraints = [0] * n_cat + mono_num

    # ===== ここから：新しいほど重くする sample_weight =====
    # 年月を「通算月」に変換（最新月との差＝どれだけ古いか）
    year_s  = pd.to_numeric(df["year"],  errors="coerce").astype(int)
    month_s = pd.to_numeric(df["month"], errors="coerce").astype(int)

    latest_m = int(year_s.max())*12 + int(month_s.max())          # スカラ
    this_m   = (year_s*12 + month_s)                               # Series（ベクトル計算）
    months_old = (latest_m - this_m).clip(lower=0)

    HALF_LIFE_MONTHS = 12.0  # 半減期（例：12ヶ月）。強めに効かせたいなら 6 にする等
    w = np.power(0.5, months_old / HALF_LIFE_MONTHS)
    w = np.clip(w, 0.2, None)  # 古すぎるデータにも最低限の重み（お好みで調整）
    sample_weight = w.values.astype(float)
    # ===== ここまで =====

    # ===== エンコードして学習用テーブル =====
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_cat = oe.fit_transform(df[feat_cat].fillna("NA")).astype("int32")
    X_num = df[feat_num_all].astype(float).values
    X = np.hstack([X_cat, X_num])

    cat_cols = [f"cat__{c}" for c in feat_cat]
    num_cols = feat_num_all.copy()
    X_df = pd.DataFrame(X, columns=cat_cols + num_cols)
    categorical_feature = cat_cols

    meta = {
        "feat_cat": feat_cat,
        "feat_num": num_cols,
        "categorical_feature": categorical_feature,
        "monotone_constraints": monotone_constraints,
    }

    # ===== 推論用マップ（station→area を含めて保存）=====
    infer_maps = {
        "station_count_map": station_cnt.to_dict(),
        "layout_count_map":  layout_cnt.to_dict(),
        "area_count_map":    area_cnt.to_dict(),

        "station_median_log_map": station_med_map,
        "layout_median_log_map":  layout_med_map,
        "area_median_log_map":    area_med_map,

        "global_station_median_log": global_station_med,
        "global_layout_median_log":  global_layout_med,
        "global_area_median_log":    global_area_med,

        # ↓↓↓ 追加（平均）
        "station_mean_log_map": station_mean_map,
        "layout_mean_log_map":  layout_mean_map,
        "area_mean_log_map":    area_mean_map,

        "global_station_median_log": global_station_med,
        "global_layout_median_log":  global_layout_med,
        "global_area_median_log":    global_area_med,

        # （必要なら平均の全体値も）
        "global_station_mean_log": global_station_mean,
        "global_layout_mean_log":  global_layout_mean,
        "global_area_mean_log":    global_area_mean,

        "station_to_area_map": {s: station_to_area(s) for s in sorted(df["station"].astype(str).unique())},
        "area_groups": AREA_LIST,

        "feat_cat": feat_cat,
        "feat_num": num_cols,

        # 駅→エリア（推論で駅入力から自動付与）
        "station_to_area_map": {s: station_to_area(s) for s in sorted(df["station"].astype(str).unique())},
        "area_groups": AREA_LIST,

        "feat_cat": feat_cat,
        "feat_num": num_cols,
    }

    avg_psm_arr = df["avg_psm_yen"].to_numpy() if "avg_psm_yen" in df.columns else None
    period_arr = df["period"].to_numpy().astype(int)

    # 返り値（従来どおり）
    return (X_df, y_log, categorical_feature, monotone_constraints, meta, oe,infer_maps, sample_weight, avg_psm_arr,np.log1p(df["price_yen"].astype(float).values),period_arr)

In [191]:
# ============================
# 2) Optuna でハイパラ探索（時系列CV）
# ============================
def tune_with_optuna(
    X_df, y_log, y_log_true, categorical_feature, monotone_constraints,
    period_arr,  # ★追加（YYYYMMの配列）
    n_trials=200, sample_weight=None, avg_psm=None
) -> tuple[dict, int, float]:

    from sklearn.model_selection import TimeSeriesSplit  # 先頭に移しても可

    # 並び替え（古い→新しい）
    order = np.argsort(period_arr)
    X_ord = X_df.iloc[order]
    y_ord = y_log[order]
    y_true_ord = y_log_true[order]
    w_ord = None if sample_weight is None else sample_weight[order]
    avg_ord = None if avg_psm is None else avg_psm[order]

    tscv = TimeSeriesSplit(n_splits=5)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "objective": "rmse", "metric": "rmse",
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 5.0),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "verbosity": -1, "seed": SEED,
            "monotone_constraints": monotone_constraints,
        }

        rmsles, best_iters = [], []
        for tr_rel, va_rel in tscv.split(X_ord):
            X_tr, X_va = X_ord.iloc[tr_rel], X_ord.iloc[va_rel]
            y_tr, y_va = y_ord[tr_rel], y_ord[va_rel]
            w_tr = None if w_ord is None else w_ord[tr_rel]
            w_va = None if w_ord is None else w_ord[va_rel]

            dtr = lgb.Dataset(
                X_tr, label=y_tr, weight=w_tr,
                categorical_feature=categorical_feature, free_raw_data=True
            )
            dva = lgb.Dataset(
                X_va, label=y_va, weight=w_va,
                categorical_feature=categorical_feature, free_raw_data=True
            )

            model = lgb.train(
                params, dtr, valid_sets=[dva],
                num_boost_round=4000,
                callbacks=[lgb.early_stopping(200, verbose=False)]
            )
            pred_log = model.predict(X_va, num_iteration=model.best_iteration)

            if USE_DEFLATE and (avg_ord is not None):
                # 相対ログ + 相場ログ → 円に戻してRMSLE
                yhat_yen  = np.expm1(pred_log + np.log(avg_ord[va_rel]))
                ytrue_yen = np.expm1(y_true_ord[va_rel])
                rmsle = float(np.sqrt(mean_squared_log_error(ytrue_yen, yhat_yen)))
            else:
                rmsle = float(np.sqrt(mean_squared_log_error(np.expm1(y_va), np.expm1(pred_log))))

            rmsles.append(rmsle)
            best_iters.append(model.best_iteration)

        trial.set_user_attr("best_iters", best_iters)
        return float(np.mean(rmsles))

    study = optuna.create_study(direction="minimize",sampler=TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    best_iters_list = study.best_trial.user_attrs.get("best_iters", [1000])
    best_round = int(np.clip(np.mean(best_iters_list), 100, 4000))
    best_score = study.best_value
    return best_params, best_round, best_score

In [192]:
# ============================
# 3) 最終学習（ポイント & 分位点）
# ============================

def train_final_models(
    X_df: pd.DataFrame,
    y_log: np.ndarray,
    categorical_feature: list[str],
    monotone_constraints: list[int],
    best_params: dict,
    best_round: int,
    sample_weight: np.ndarray | None = None  # ★ 追加
):
    base_params = dict(best_params)
    base_params.update({
        "objective": "rmse",
        "metric": "rmse",
        "verbosity": -1,
        "seed": SEED,
        # ★ ポイントモデルでは単調性制約を使う
        "monotone_constraints": monotone_constraints,
    })

    # 生データを解放しない（categorical_feature を後で使えるように）
    dtrain = lgb.Dataset(
        X_df,
        label=y_log,
        weight=sample_weight,
        categorical_feature=categorical_feature,
        free_raw_data=False
    )

    # ---- ポイント（対数価格）----
    model_point = lgb.train(base_params, dtrain, num_boost_round=best_round)

    # ---- smearing（log1p → 円の戻し補正）----
    pred_log_tr = model_point.predict(X_df, num_iteration=getattr(model_point, "best_iteration", None))
    resid = y_log - pred_log_tr  # eps = y_log - yhat_log
    # ★ 直近重視の設計と整合：重み付き平均（重みが無いときは従来どおり）
    smearing = (
        float(np.average(np.exp(resid), weights=sample_weight))
        if sample_weight is not None else
        float(np.mean(np.exp(resid)))
    )

    # ---- 分位点（q10/q90）----
    models_q = {}
    for alpha, tag in [(0.1, "q10"), (0.9, "q90")]:
        q_params = dict(best_params)
        q_params.update({
            "objective": "quantile",
            "alpha": alpha,
            "metric": "quantile",
            "verbosity": -1,
            "seed": SEED,
            # ★ ここが重要：quantile では単調性制約を外す
            # "monotone_constraints": monotone_constraints,  # ←入れない
        })
        models_q[tag] = lgb.train(q_params, dtrain, num_boost_round=best_round)

    return model_point, models_q, smearing


In [193]:
# ===== 入力CSV→中間CSVの自動生成（なければ作る） =====
from pathlib import Path
import pandas as pd, numpy as np, re
from datetime import datetime

RAW_CSV = Path("/content/Tokyo_Chuo Ward_20242_20251.csv")
IN = Path("data/interim/train_tabular.csv")
ST_OUT = Path("data/interim/stations.csv")
IN.parent.mkdir(parents=True, exist_ok=True)

# ===== _read_chuo_and_save() の先頭付近に追加（関数の外でOK）=====
def _parse_period(s):
    """'2025年8月', '2025/08', '2025-8', '2025年第2四半期' などを (year, month) に直す"""
    import re, math
    if pd.isna(s): return (np.nan, np.nan)
    s = str(s)

    # 年月パターン
    m = re.search(r"(\d{4})[年/\-\. ]+(\d{1,2})", s)
    if m:
        y, mo = int(m.group(1)), int(m.group(2))
        mo = int(np.clip(mo, 1, 12))
        return (y, mo)

    # 四半期パターン
    m = re.search(r"(\d{4})年?第?([1-4])四半期", s)
    if m:
        y, q = int(m.group(1)), int(m.group(2))
        q2m = {1:1, 2:4, 3:7, 4:10}
        return (y, q2m[q])

    # 年だけ
    m = re.search(r"(\d{4})年", s)
    if m:
        return (int(m.group(1)), 1)

    return (np.nan, np.nan)

def _read_chuo_and_save(raw_csv: Path, out_csv: Path, stations_csv: Path):
    # エンコーディングを順に試す（Excel想定）
    last_err = None
    for enc in ("cp932", "utf-8-sig", "utf-8", "utf-16", "utf-16le", "utf-16be"):
        try:
            raw = pd.read_csv(raw_csv, encoding=enc)
            break
        except Exception as e:
            last_err = e
            raw = None
    if raw is None:
        raise last_err

    def _to_year(x):
        if pd.isna(x):
            return np.nan
        m = re.search(r"(\d{4})年", str(x))
        return int(m.group(1)) if m else np.nan

    # 必要列だけ取り出し
    df = raw[["最寄駅：名称","最寄駅：距離（分）","間取り","面積（㎡）","建築年","取引価格（総額）"]].copy()

    build_year = df["建築年"].apply(_to_year)
    current_year = datetime.now().year
    df["築年数"] = (current_year - build_year).clip(lower=0)

    for col in ["最寄駅：距離（分）","面積（㎡）","築年数"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(df[col].median())

    def _normalize_layout(s):
        if not isinstance(s, str):
            return "その他"
        s = (s.upper()
               .replace("Ｌ","L").replace("Ｄ","D").replace("Ｋ","K").replace("Ｒ","R"))
        m = re.search(r"(\d+)(LDK|DK|K|R)", s)
        return m.group(0) if m else "その他"

    # ★ ここは _normalize_layout の「外」（= 同じインデント階層）
    period_col = None
    for c in ["取引時期", "取引時点", "成約年月", "契約年月", "年月", "取引年月"]:
        if c in raw.columns:
            period_col = c
            break

    if period_col is not None:
        ym = raw[period_col].apply(_parse_period)
        year  = ym.apply(lambda t: t[0]).astype("Int64")
        month = ym.apply(lambda t: t[1]).astype("Int64")
    else:
        year  = pd.Series([np.nan] * len(raw), dtype="Int64")
        month = pd.Series([np.nan] * len(raw), dtype="Int64")

    # 欠損処理
    if year.isna().all():
        now = datetime.now()
        year  = pd.Series([now.year]  * len(raw))
        month = pd.Series([now.month] * len(raw))
    else:
        year  = year.fillna(method="ffill").fillna(method="bfill").astype(int)
        month = month.fillna(method="ffill").fillna(method="bfill").astype(int)

    # 学習用テーブル作成
    train = df.rename(columns={
        "最寄駅：名称": "station",
        "最寄駅：距離（分）": "walk_min",
        "面積（㎡）": "area_sqm",
        "取引価格（総額）": "price_yen",
    })[["station","walk_min","築年数","area_sqm","price_yen"]].copy()

    train["layout"] = df["間取り"].astype(str).apply(_normalize_layout)
    train["year"]   = year.astype(int)
    train["month"]  = month.astype(int)
    train["period"] = train["year"] * 100 + train["month"]

    # 保存
    train.to_csv(out_csv, index=False)

    # 駅リスト保存（utf-8-sig）
    s = (raw["最寄駅：名称"].astype(str)
           .replace({"nan": np.nan, "None": np.nan, "": np.nan})
           .dropna()
           .str.normalize("NFKC")
           .str.replace("（","(",regex=False).str.replace("）",")",regex=False)
           .str.replace("　"," ",regex=False).str.strip()
           .str.replace(r"\s+"," ",regex=True))
    pd.Series(sorted(s.unique())).to_csv(stations_csv, index=False, header=False, encoding="utf-8-sig")

# まだ中間CSVがない場合は作る
# if not IN.exists():
#     if not RAW_CSV.exists():
#         raise FileNotFoundError(f"Raw CSV が見つかりません: {RAW_CSV.resolve()}")
#     _read_chuo_and_save(RAW_CSV, IN, ST_OUT)
# ★ 一度だけ True にしてCSVを作り直す（year/month/period列を付与）
FORCE_REBUILD = True   # 生成が終わったら False に戻す/この行を消す

if FORCE_REBUILD or not IN.exists():
   if not RAW_CSV.exists():
      raise FileNotFoundError(f"Raw CSV が見つかりません: {RAW_CSV.resolve()}")
   _read_chuo_and_save(RAW_CSV, IN, ST_OUT)

/usr/local/lib/python3.12/dist-packages/cudf/pandas/fast_slow_proxy.py:28: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/cudf/pandas/fast_slow_proxy.py:28: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/cudf/pandas/fast_slow_proxy.py:28: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/cudf/pandas/fast_slow_proxy.py:28: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return fn(*args, **kwargs)


In [194]:
def _ensure_pop_column(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    gdf に 'pop' 列が無い場合、人口らしい数値列を自動検出して 'pop' に採用する。
    ルール: (名前ヒントの優先度, -合計値) のタプルで最小の列を選ぶ。合計0の列は不採用。
    """
    if "pop" in gdf.columns:
        gdf = gdf.copy()
        gdf["pop"] = pd.to_numeric(gdf["pop"], errors="coerce").fillna(0.0)
        return gdf

    # 数値列候補（geometryや明らかなID/座標は除外）
    ignore_like = {"geometry","mesh","meshcode","code","id","lat","lon","x","y"}
    num_cols = [c for c in gdf.columns
                if c not in ignore_like
                and c != "geometry"
                and np.issubdtype(gdf[c].dtype, np.number)]

    # 文字列でも数値化できるものは候補に含める（CSV→GeoJSON時の型ブレ対策）
    for c in gdf.columns:
        if c in ignore_like or c == "geometry" or c in num_cols:
            continue
        # “数値化できる比率”が高い列も候補に
        try:
            v = pd.to_numeric(gdf[c], errors="coerce")
            if v.notna().mean() > 0.9:
                num_cols.append(c)
        except Exception:
            pass

    if not num_cols:
        raise ValueError("人口候補となる数値列が見つかりません（pop作成失敗）。")

    name_hints = ["総人口","将来","future","pop","population","ppl","pt00","ptn"]
    def name_rank(col):
        s = str(col).lower()
        # 列名そのものと低頻度の誤表記も拾う
        for i, k in enumerate(name_hints):
            if k in s:
                return i
        return 999

    best = None
    best_col = None
    for c in num_cols:
        vals = pd.to_numeric(gdf[c], errors="coerce").fillna(0.0)
        total = float(vals.sum())
        if total <= 0:
            continue
        score = (name_rank(c), -total)  # 名前優先→合計が大きいほど良い
        if best is None or score < best:
            best = score
            best_col = c

    if best_col is None:
        raise ValueError("人口候補列の合計がすべて0でした（pop作成失敗）。")

    gdf = gdf.copy()
    gdf["pop"] = pd.to_numeric(gdf[best_col], errors="coerce").fillna(0.0)
    print(f"[INFO] Using '{best_col}' as population -> created 'pop' column (sum={gdf['pop'].sum():.1f}).")
    return gdf


In [195]:
from pathlib import Path

FLOOD_GJ = Path("data/external/flood_zone.geojson")
POP_GJ   = Path("data/external/popmesh_250m.geojson")  # 今回は「点（メッシュ中心）」をGeoJSON化

# 公式から取得（初回DL → 以降はキャッシュ）
_ = fetch_ksj_layer("A49",       "13", FLOOD_GJ)      # 高潮（東京都）
_ = fetch_ksj_layer("mesh250r6", "13", POP_GJ)        # 将来推計人口250m（東京都）

g_pop = gpd.read_file(POP_GJ)
g_pop = _ensure_pop_column(g_pop)
print(g_pop.head())

# 以降は既存の関数で 400m 特徴量へ
_ = build_station_env_features(
    stations_geo_csv=STATIONS_GEO_CSV,  # data/external/stations_geo.csv（①で配置）
    flood_geojson=FLOOD_GJ,             # ポリゴン
    pop_geojson=POP_GJ,                 # “点”だが weighted 合計は build 側でOK
    out_csv=ENV_BY_ST,
    buffer_m=400.0
)
print("環境特徴を保存:", ENV_BY_ST)




[INFO] Using '0' as population -> created 'pop' column (sum=99740364880153.0).
            0           1                    geometry         pop
0  3653375813    4.228446  POINT (153.98438 24.32292)  3653375813
1  3741123523  207.111769  POINT (141.29688 24.82292)  3741123523
2  3942715342         0.0  POINT (142.20312 26.65625)  3942715342
3  3942716241   57.424601  POINT (142.23438 26.63542)  3942716241
4  3942716242   11.185976  POINT (142.23438 26.63542)  3942716242
[INFO] Using '0' as population -> created 'pop' column (sum=99740364880153.0).


/tmp/ipython-input-10174556.py:208: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  union_buf = g_st_3857["buf"].unary_union
/tmp/ipython-input-10174556.py:220: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  flood_union = g_flood_3857.unary_union  # 距離計算用（高速）


KeyError: 'station'

In [ ]:
g = gpd.read_file(POP_GJ)
g = _ensure_pop_column(g)
print("POP_GJ features:", len(g))
print("Total pop sum:", float(pd.to_numeric(g["pop"], errors="coerce").fillna(0).sum()))

In [ ]:
tmp = pd.read_csv(IN)
print(tmp.columns.tolist())
# 'year', 'month', 'period' が含まれていること
print(tmp.head())

In [ ]:
# ============================
# 4) 実行
# ============================

if __name__ == "__main__":
    assert IN.exists(), f"Not found: {IN}"

    # ★ infer_maps まで受け取る（関数が infer_maps を返す実装になっている前提）
    X_df, y_log, cat_cols, mono_cons, meta, oe, infer_maps, sample_weight, avg_psm_arr, y_log_true, period_arr = build_table_and_features()

    print(f"Train rows: {len(X_df):,}, cols: {X_df.shape[1]} (cats={len(cat_cols)})")

    best_params, best_round, best_score = tune_with_optuna(
    X_df, y_log, y_log_true, cat_cols, mono_cons, period_arr,   # ★ period_arr を追加
    n_trials=200, sample_weight=sample_weight, avg_psm=avg_psm_arr
)
    print("[OPTUNA] best RMSLE:", round(best_score, 4))
    print("[OPTUNA] best params:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    print("[OPTUNA] best_round:", best_round)

    model_point, models_q, smearing = train_final_models(
        X_df, y_log, cat_cols, mono_cons, best_params, best_round,
    sample_weight=sample_weight
    )

    # 直近水準の倍率キャリブを外部指数から計算（保存の前！）
    calib = compute_calibration(period_arr, IDX_CSV, smooth_months=2)
    print(f"[CALIB] multiplier to latest: {calib:.4f}")

    # ===== 保存 =====
    # 予測時に使うメタ情報（列順や制約）とエンコーダを一緒に保存しておく
    dump({
        "model": model_point,
        "encoder": oe,
        "cat_cols": meta["feat_cat"],
        "num_cols": meta["feat_num"],
        "categorical_feature": meta["categorical_feature"],
        "monotone_constraints": meta["monotone_constraints"],
        "target": "log_price_yen",
        "smearing": smearing,
        "calibration": calib,          # ★追加
        "use_deflate": USE_DEFLATE,    # ★追加（推論コード用に）
    }, OUT_DIR / "lgbm_optuna_point.pkl")

    dump({
        "model": models_q["q10"],
        "encoder": oe,
        "cat_cols": meta["feat_cat"],
        "num_cols": meta["feat_num"],
        "categorical_feature": meta["categorical_feature"],
        "monotone_constraints": meta["monotone_constraints"],
        "target": "log_price_yen",
        "alpha": 0.1,
        "smearing": smearing,
        "calibration": calib,          # ★追加
        "use_deflate": USE_DEFLATE,    # ★追加（推論コード用に）
    }, OUT_DIR / "lgbm_optuna_q10.pkl")

    dump({
        "model": models_q["q90"],
        "encoder": oe,
        "cat_cols": meta["feat_cat"],
        "num_cols": meta["feat_num"],
        "categorical_feature": meta["categorical_feature"],
        "monotone_constraints": meta["monotone_constraints"],
        "target": "log_price_yen",
        "alpha": 0.9,
        "smearing": smearing,
        "calibration": calib,          # ★追加
        "use_deflate": USE_DEFLATE,    # ★追加（推論コード用に）
    }, OUT_DIR / "lgbm_optuna_q90.pkl")

    with open(OUT_DIR / "feature_config.json", "w", encoding="utf-8") as f:
        json.dump({
            "feat_cat": meta["feat_cat"],
            "feat_num": meta["feat_num"],
            "categorical_feature": meta["categorical_feature"],
            "monotone_constraints": meta["monotone_constraints"],
            "notes": "y is log1p(price_yen). During inference, apply expm1 to predictions.",
        }, f, ensure_ascii=False, indent=2)

    # ★ 推論用マップも保存（カッコを閉じるのを忘れない）
    with open(OUT_DIR / "infer_maps.json", "w", encoding="utf-8") as f:
        json.dump(infer_maps, f, ensure_ascii=False, indent=2)

    print("Saved models →", OUT_DIR)


In [ ]:
import matplotlib.pyplot as plt

# 重要度を取得
importance = model_point.feature_importance(importance_type="gain")
feature_names = model_point.feature_name()

# DataFrame化
import pandas as pd
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False)

print(fi)

# 棒グラフで可視化
plt.figure(figsize=(8, 6))
plt.barh(fi["feature"], fi["importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance (by gain)")
plt.show()

In [ ]:
# ===== ここから追加：学習＆検証（OOF）精度の表示 =====
from sklearn.metrics import r2_score
import numpy as np

def _fmt_metric_row(tag, rmsle, r2, rmse_y, mae_y, mape):
    print(f"[{tag}] RMSLE={rmsle:.4f} | R2(log)={r2:.4f} | RMSE(円)={rmse_y:,.0f} | MAE(円)={mae_y:,.0f} | MAPE={mape:.2f}%")

def _yen_metrics(y_true_yen: np.ndarray, y_pred_yen: np.ndarray):
    rmse_y = float(np.sqrt(np.mean((y_true_yen - y_pred_yen) ** 2)))
    mae_y  = float(np.mean(np.abs(y_true_yen - y_pred_yen)))
    mape   = float(np.mean(np.abs((y_true_yen - y_pred_yen) / np.clip(y_true_yen, 1.0, None))) * 100.0)
    return rmse_y, mae_y, mape

# --- 学習データ上の精度 ---
best_it = getattr(model_point, "best_iteration", None)
y_log_pred_tr = model_point.predict(X_df, num_iteration=best_it)

# deflated のままのRMSE（RMSLE相当）
rmsle_tr = float(np.sqrt(np.mean((y_log - y_log_pred_tr) ** 2)))
r2_tr    = float(r2_score(y_log, y_log_pred_tr))

# ←← 真の価格は必ず y_log_true から戻す！
y_true_yen = np.expm1(y_log_true)

try:
    smear = float(smearing)
except NameError:
    smear = 1.0

if USE_DEFLATE:
    # 予測（相対ログ）に相場ログを足してから円に戻す（★ df[...] ではなく avg_psm_arr を使う）
    yhat_log1p_tr = y_log_pred_tr + np.log(avg_psm_arr)
    y_pred_yen_tr = smear * np.expm1(yhat_log1p_tr)
else:
    y_pred_yen_tr = smear * np.expm1(y_log_pred_tr)

rmse_y_tr, mae_y_tr, mape_tr = _yen_metrics(y_true_yen, y_pred_yen_tr)
_fmt_metric_row("TRAIN", rmsle_tr, r2_tr, rmse_y_tr, mae_y_tr, mape_tr)

# ② KFold OOF（疑似バリデーション）での精度
from sklearn.model_selection import KFold
import lightgbm as lgb

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_pred_log = np.zeros_like(y_log, dtype=float)

# best_params / best_round / cat_cols / mono_cons は既存の変数を使用
for tr_idx, va_idx in kf.split(X_df):
    X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
    y_tr, y_va = y_log[tr_idx], y_log[va_idx]

     # ★ 追加：重みをfoldごとにスライス
    w_tr = None if 'sample_weight' not in globals() or sample_weight is None else sample_weight[tr_idx]
    w_va = None if 'sample_weight' not in globals() or sample_weight is None else sample_weight[va_idx]

    # ★ ここで weight=... を渡す
    dtr = lgb.Dataset(
        X_tr, label=y_tr, weight=w_tr,
        categorical_feature=cat_cols, free_raw_data=True
    )
    dva = lgb.Dataset(
        X_va, label=y_va, weight=w_va,
        categorical_feature=cat_cols, free_raw_data=True
    )

    params = dict(best_params)
    params.update({
        "objective": "rmse",
        "metric": "rmse",
        "verbosity": -1,
        "seed": SEED,
        "monotone_constraints": mono_cons,
    })

    booster = lgb.train(
        params, dtr,
        num_boost_round=best_round,
        valid_sets=[dva],
        callbacks=[],  # ここでは早停なし（best_round固定）
    )
    oof_pred_log[va_idx] = booster.predict(X_va, num_iteration=getattr(booster, "best_iteration", None))

# --- OOF（疑似バリデーション）---
# log空間のRMSE（deflated RMSE）
rmsle_oof = float(np.sqrt(np.mean((y_log - oof_pred_log) ** 2)))
r2_oof    = float(r2_score(y_log, oof_pred_log))

# 円スケール（deflate時は相場ログを足す）
if USE_DEFLATE and (avg_psm_arr is not None):
    y_pred_oof_yen = np.expm1(oof_pred_log + np.log(avg_psm_arr))
else:
    y_pred_oof_yen = np.expm1(oof_pred_log)

y_true_yen = np.expm1(y_log_true)
rmse_y_oof, mae_y_oof, mape_oof = _yen_metrics(y_true_yen, y_pred_oof_yen)
_fmt_metric_row("OOF", rmsle_oof, r2_oof, rmse_y_oof, mae_y_oof, mape_oof)

# === 直近水準の倍率キャリブ（最近 K ヶ月） ===
CALIB_RECENT_K = 2   # ← 好みで 2〜3 を推奨
uniq = np.unique(period_arr)
recent = uniq[-CALIB_RECENT_K:]
mask_recent = np.isin(period_arr, recent)
ratio = y_true_yen[mask_recent] / np.clip(y_pred_oof_yen[mask_recent], 1.0, None)
calib = float(np.median(np.clip(ratio, 0.7, 1.3)))  # クリップは安定化のため
print(f"[Calib] factor={calib:.4f} (recent {CALIB_RECENT_K} months)")

# 参考：キャリブ適用後の円スケール指標も表示
rmse_y_oof_c, mae_y_oof_c, mape_oof_c = _yen_metrics(y_true_yen, calib * y_pred_oof_yen)
_fmt_metric_row("OOF × Calib", rmsle_oof, r2_oof, rmse_y_oof_c, mae_y_oof_c, mape_oof_c)

# 直近重み付き（任意）
if 'sample_weight' in globals() and sample_weight is not None:
    # log側の重み付き誤差
    def _weighted_mean(x, w): return float(np.sum(w*x)/np.sum(w))
    rmsle_oof_w = float(np.sqrt(_weighted_mean((y_log - oof_pred_log)**2, sample_weight)))
    # 円スケール重み付き
    def _yen_metrics_weighted(y_true_yen, y_pred_yen, w):
        rmse_y = float(np.sqrt(_weighted_mean((y_true_yen - y_pred_yen)**2, w)))
        mae_y  = float(_weighted_mean(np.abs(y_true_yen - y_pred_yen), w))
        mape   = float(_weighted_mean(np.abs((y_true_yen - y_pred_yen) / np.clip(y_true_yen, 1.0, None)), w) * 100.0)
        return rmse_y, mae_y, mape
    rmse_y_oof_w, mae_y_oof_w, mape_oof_w = _yen_metrics_weighted(y_true_yen, y_pred_oof_yen, sample_weight)
    rmse_y_oof_w_c, mae_y_oof_w_c, mape_oof_w_c = _yen_metrics_weighted(y_true_yen, calib * y_pred_oof_yen, sample_weight)
    _fmt_metric_row("OOF[w]",   rmsle_oof_w, r2_oof, rmse_y_oof_w,   mae_y_oof_w,   mape_oof_w)
    _fmt_metric_row("OOF[w]×C", rmsle_oof_w, r2_oof, rmse_y_oof_w_c, mae_y_oof_w_c, mape_oof_w_c)